# Aula 08 · JSON, datas e exceções

Este é o caderno da aula de hoje, e ele vem **sem as saídas** de propósito.
Em cada exemplo, o caminho é sempre o mesmo:

1. **leia** o código, sem rodar;
2. **escreva** na célula `_Sua previsão:_`, logo abaixo dele, o que você acha
   que vai sair;
3. **rode** a célula do código e compare com o que você escreveu;
4. **abra** o `▶ O que aconteceu` para ler a explicação.

A previsão errada é a parte que ensina — não a apague.

**Ao fim desta aula você deve conseguir:**

1. converter JSON em estrutura Python e de volta, e reconhecer o que torna um JSON inválido;
2. calcular duração entre dois timestamps e explicar por que comparar datas como texto não serve;
3. proteger o processamento de um item com `try/except` específico, registrando o descarte.

## Preparação

Rode esta célula primeiro.

In [ ]:
import json
from datetime import datetime, timedelta

## Parte 1 — Exemplos: prever, rodar, investigar

### Q1. JSON vira dicionário e lista

_Leia sem rodar._

In [ ]:
texto = '{"coleta": "2026-03-02", "equipamentos": [{"nome": "OLT-A", "portas": 16}]}'
dados = json.loads(texto)
print(type(dados))
print(dados["coleta"])
print(dados["equipamentos"][0]["nome"])
print(len(dados["equipamentos"]))

**Preveja:** "que tipo o `loads` devolve?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`json.loads` transforma o texto JSON em estruturas Python: objeto vira dicionário,
lista vira lista, `true` vira `True` e `null` vira `None`. Depois disso não há nada
novo — `dados["equipamentos"][0]["nome"]` é a mesma navegação da Aula 06, sobre uma
lista de dicionários.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#json)

</details>

### Q2. E o caminho de volta

_Leia sem rodar._

In [ ]:
print(json.dumps({"total": 2, "em_servico": 1}))

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`json.dumps` faz o caminho de volta: de estrutura Python para texto JSON. É o que
você usa para gravar um resumo em arquivo ou enviá-lo a outro sistema. Com
`indent=2` sai legível para humano, e com `ensure_ascii=False` os acentos saem como
acentos em vez de `\u00e7`.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#json)

</details>

### Q3. Data em texto × data de verdade

_Leia sem rodar._

In [ ]:
a = "2026-03-02 09:00:00"
b = "2026-03-02 14:03:17"
print(a < b)

inicio = datetime.strptime(a, "%Y-%m-%d %H:%M:%S")
fim = datetime.strptime(b, "%Y-%m-%d %H:%M:%S")
print(fim - inicio)
print((fim - inicio).total_seconds())
print(round((fim - inicio).total_seconds() / 60, 1))
print(inicio + timedelta(minutes=90))
print(inicio.strftime("%d/%m/%Y %H:%M"))

**Preveja:** "a primeira linha funciona? E `b - a`, funcionaria?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Um timestamp em texto serve para ler e, às vezes, comparar — mas **nunca para
calcular**. `strptime` o converte em `datetime`, e aí a subtração de duas datas dá
uma **duração** (`timedelta`), que `.total_seconds()` transforma em número. Esse
quarteto — ler, subtrair, comparar e formatar com `strftime` — é tudo que se precisa
para MTTR, janela de indisponibilidade e intervalo coberto por um log.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#datas)

</details>

### Q3b. Investigue: e se a data estivesse no formato brasileiro?

_Leia sem rodar._

In [ ]:
# 31 de dezembro vem ANTES de 1 de janeiro -- mas nao em ordem alfabetica.
print("31/12/2025" < "01/01/2026")
print(datetime.strptime("31/12/2025", "%d/%m/%Y") < datetime.strptime("01/01/2026", "%d/%m/%Y"))

**Preveja:** "31 de dezembro vem antes de 1º de janeiro. As duas linhas concordam?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

não concordam. Comparadas **como texto**, as datas são lidas da
esquerda para a direita: `"3"` é maior que `"0"`, então o réveillon "acontece
depois" do ano-novo. Convertidas em `datetime`, a comparação é cronológica e acerta.
O formato `AAAA-MM-DD` do log do curso não tem esse problema — mas qualquer data
vinda de planilha brasileira tem.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#datas)

</details>

### Q4. `try/except` **por item**

_Leia sem rodar._

In [ ]:
linhas = [
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal",
    "",
    "linha truncada",
    "2026-03-02 10:02:55 CRITICAL ONU-SUL-4512 sem resposta",
]
print(len(linhas))

**Preveja:** "quantos críticos e quantas ignoradas?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Este é o log que chega na vida real: quatro linhas, das quais duas não têm o formato
esperado — uma está vazia e a outra está truncada. Qualquer `linha.split()[2]` nelas
levanta `IndexError`. A pergunta da próxima célula é o que fazer com isso.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#excecoes-o-coletor-que-nao-morre)

</details>

### Q4b. `try/except` por item: a linha ruim cai, o resto continua

_Leia sem rodar._

In [ ]:
criticos = 0
ignoradas = 0
for linha in linhas:
    try:
        if linha.split()[2] == "CRITICAL":
            criticos = criticos + 1
    except IndexError:
        ignoradas = ignoradas + 1
print(criticos, ignoradas)

**Preveja:** "quantos críticos e quantas ignoradas?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

o `try` está **dentro** do laço, envolvendo o processamento de uma
linha — e é essa a decisão que importa. Assim, a linha malformada é descartada e o
laço continua, entregando os dois alarmes válidos e a contagem de dois descartes. O
programa não morre **e não finge**: ele informa as duas coisas.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#excecoes-o-coletor-que-nao-morre)

</details>

### Q4c. Investigue: e se o `try` envolvesse o laço inteiro?

_Leia sem rodar._

In [ ]:
criticos_frageis = 0
try:
    for linha in linhas:
        if linha.split()[2] == "CRITICAL":
            criticos_frageis = criticos_frageis + 1
except IndexError:
    pass
print(criticos_frageis)

**Preveja:** "quantos críticos esta versão conta?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

conta **um**. Ao encontrar a segunda linha, o erro sobe, o `except`
captura, e o laço inteiro é abandonado — o alarme crítico que estava na quarta linha
nunca é visto. É a diferença entre **perder uma linha** e **perder o arquivo**, e é
por isso que o `try` vai dentro do laço.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#excecoes-o-coletor-que-nao-morre)

</details>

### Q5. O `except` que engole o problema

_Leia sem rodar._

In [ ]:
criticos_mudos = 0
for linha in linhas:
    try:
        if linha.split()[2] == "CRITICAL":
            criticos_mudos = criticos_mudos + 1
    except IndexError:
        pass
print(criticos_mudos)

**Preveja:** "esta versão dá um resultado diferente da anterior?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

O resultado é **o mesmo** da célula anterior — e é justamente esse o problema. O
`except: pass` descarta as duas linhas ruins em silêncio, e ninguém fica sabendo. Um
`except` que não registra nada transforma "processei 998 de 1000 linhas" em
"processei tudo, tá ótimo". **Capture o erro específico e registre o descarte.**

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#excecoes-o-coletor-que-nao-morre)

</details>

### Q6. Conferir antes × proteger com `try`

_Leia sem rodar._

In [ ]:
def para_numero(texto):
    """Converte para float, devolvendo None quando nao da."""
    try:
        return float(texto)
    except ValueError:
        return None

print(para_numero("-21.4"))
print(para_numero("sem sinal"))
print(para_numero(""))

**Preveja:** "as três linhas."

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Uma função pequena que **tenta converter e devolve `None` quando não dá** é o
formato que você vai repetir bastante. Quem chama decide o que fazer com o `None` —
e é sempre mais fácil decidir isso de fora da função do que dentro dela. Repare que
a string vazia também falha na conversão, e não só o texto sem sentido.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#excecoes-o-coletor-que-nao-morre)

</details>

_Anotações da Parte 1:_

## Parte 2 — Resolver junto

Tente sozinho primeiro, por cinco minutos, na sua máquina. Depois
resolvemos juntos.

### E1. Ler o JSON da coleta

A resposta de um sistema de gerência chega como texto JSON:

```json
{"coleta": "2026-03-02", "equipamentos": [{"nome": "OLT-A", "portas": 16}]}
```

Escreva `equipamentos(texto)`, que converte o texto e devolve a **lista** de
equipamentos.

```python
equipamentos('{"coleta": "2026-03-02", "equipamentos": [{"nome": "OLT-A"}]}')
# -> [{"nome": "OLT-A"}]
```

Se a chave `"equipamentos"` não existir, devolva lista vazia.

**Assinatura:**

```python
import json

def equipamentos(texto):
    """Lista de equipamentos contida no JSON recebido."""
```

In [ ]:
# sua solução aqui


### E2. Momento e duração

Escreva `momento(texto)`, que converte um timestamp no formato
`"AAAA-MM-DD HH:MM:SS"` para `datetime`. Se o texto não estiver nesse formato,
devolva `None` — **sem deixar o erro subir**.

```python
momento("2026-03-02 14:03:17")   # -> datetime(2026, 3, 2, 14, 3, 17)
momento("02/03/2026")            # -> None
momento("")                      # -> None
```

**Assinatura:**

```python
from datetime import datetime

def momento(texto):
    """Converte o timestamp para datetime, ou None se o formato não bater."""
```

Escreva `duracao_minutos(inicio, fim)`, que recebe dois timestamps em texto e
devolve a duração em **minutos**, com uma casa decimal.

```python
duracao_minutos("2026-03-02 14:03:17", "2026-03-02 15:48:17")   # -> 105.0
```

Se qualquer um dos dois não estiver no formato esperado, devolva `None`.

**Assinatura:**

```python
from datetime import datetime

def duracao_minutos(inicio, fim):
    """Minutos entre os dois timestamps, ou None se algum for inválido."""
```

In [ ]:
# sua solução aqui


### E3. A média que informa o descarte

Escreva `media_valida(textos)`, que recebe uma lista de leituras em texto e
devolve a tupla `(media, descartadas)`: a média dos valores que dava para
converter (duas casas) e **quantos** foram descartados.

```python
media_valida(["-21.4", "sem sinal", "-19.8"])   # -> (-20.6, 1)
media_valida(["erro", "erro"])                  # -> (0.0, 2)
media_valida([])                                # -> (0.0, 0)
```

Este é o princípio da unidade: o programa não morre por causa de uma leitura ruim,
mas também **não finge que ela não existiu**.

**Assinatura:**

```python
def media_valida(textos):
    """Devolve (média das leituras válidas, quantidade de descartadas)."""
```

In [ ]:
# sua solução aqui


### E4. O coletor completo

Feche a unidade com o coletor completo. Escreva `processa_log(linhas)`, que
devolve um **dicionário** com o resumo do log:

```python
{
    "processadas": 3,        # linhas em formato válido
    "ignoradas": 2,          # linhas descartadas
    "criticos": 2,           # alarmes CRITICAL entre as processadas
    "equipamentos": 2,       # equipamentos distintos que apareceram
    "minutos": 105.0,        # intervalo coberto pelo log
}
```

Para um log sem nenhuma linha válida, todos os números são `0` (e `minutos` é
`0.0`).

É o mesmo relatório que o projeto final vai produzir — e repare que ele informa
**as duas coisas**: o que foi processado e o que foi descartado.

**Assinatura:**

```python
from datetime import datetime

FORMATO = "%Y-%m-%d %H:%M:%S"

def momento_da_linha(campos):
    """Devolve o datetime dos dois primeiros campos, ou None. JÁ ESCRITA."""
    try:
        return datetime.strptime(f"{campos[0]} {campos[1]}", FORMATO)
    except ValueError:
        return None

def processa_log(linhas):
    """Resumo do log: processadas, ignoradas, críticos, equipamentos e minutos."""
    #   1. separe os campos; se forem menos de 5, é ignorada -> continue
    #   2. chame momento_da_linha(campos); se devolver None, é ignorada -> continue
    #   3. senão, conte como processada, guarde o momento numa lista, guarde o
    #      equipamento (campos[3]) num conjunto, e conte se campos[2] é CRITICAL
    # No fim, os minutos saem de max(momentos) - min(momentos).
```

In [ ]:
# sua solução aqui


_Anotações da Parte 2:_

## Parte 3 — Quiz de conceitos

Responda de cabeça, sem rodar. Conferimos juntos no fim.

**1.** `json.loads('{"portas": 16}')` devolve:

a) uma string  b) um dicionário  c) uma lista  d) um objeto JSON

**2.** É JSON válido:

a) `{'nome': 'OLT-A'}`  b) `{"ativo": True}`  c) `{"ativo": true}`  d) `{nome: "OLT-A"}`

**3.** Subtrair dois `datetime` devolve:

a) um número de segundos  b) um `timedelta`  c) uma string  d) erro

**4.** O `try/except` que envolve o **laço inteiro**, num arquivo cuja terceira linha está malformada:

a) processa tudo  b) processa até a segunda linha e para  c) pula só a terceira  d) não processa nada

**5.** `except: pass` é má prática porque:

a) é mais lento  b) esconde qualquer erro, inclusive os do seu próprio código  c) não compila  d) só funciona com `ValueError`